In [1]:
import sklearn
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
#no wrapping
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)
pd.set_option('display.expand_frame_repr', False)  # Disable line-wrapping
pd.set_option('display.max_rows', 10)

In [5]:
#coffee_clim_data = pd.read_csv(r"..\data\coll_caff_node_clim.csv")
#coffee_env_data = pd.read_csv(r"..\data\coll_caff_node_env.csv")
coffee_data = pd.read_csv(r"..\data\coll_all_new_jan.csv")

caffeine_content = pd.read_csv(r"..\input\no_caffeine_nodes_w_specimen.csv")

coffee_data.head

<bound method NDFrame.head of             specimen_id  source_crs  longitude   latitude                              mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded         specimen_name  clim_1_tmin1_jan  ...  env_73_asp  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old
0           abbayesii_0        4326  46.450000 -24.366660  POINT (647070.6204618097 7304410.220689219)              79              0                   False      Coffea_abbayesii             195.5  ...       307.0         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0
1           abbayesii_1        4326  46.830000 -24.730000  POINT (685086.8720605411 7263711.610572983)              79              0                   False      Coffea_abbayesii             177.5  ...        56.0         7630.0        10.0       

In [7]:
caffeine_content

,Species_name,caffeine_percent
0,C_andrambovatensis_A310,0.000
1,C_abbayesii_A601,0.000
2,C_arenesiana_A403,0.000
3,C_bertrandii_A5,0.000
4,C_dubardii_A969,0.000
...,...,...
20,C_vianneyi_A946,0.040
21,C_farafanganensis_A208,0.045
22,C_homollei_A945,0.060
23,C_kianjavatensis_A602,0.700


In [6]:
coffee_data.drop(coffee_data.columns[0], axis=1, inplace=True)
coffee_data.head

<bound method NDFrame.head of      source_crs  longitude   latitude                              mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded         specimen_name  clim_1_tmin1_jan  clim_2_tmin2_feb  ...  env_73_asp  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old
0          4326  46.450000 -24.366660  POINT (647070.6204618097 7304410.220689219)              79              0                   False      Coffea_abbayesii             195.5             195.0  ...       307.0         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0
1          4326  46.830000 -24.730000  POINT (685086.8720605411 7263711.610572983)              79              0                   False      Coffea_abbayesii             177.5             177.0  ...        56.0         7630.0        10.0         9.0 

In [16]:
caffeine_content

,Species_name,caffeine_percent
0,C_andrambovatensis_A310,0.000
1,C_abbayesii_A601,0.000
2,C_arenesiana_A403,0.000
3,C_bertrandii_A5,0.000
4,C_dubardii_A969,0.000
5,C_heimii_A516,0.000
6,C_humbertii_RNF785,0.000
7,C_millotii_A222,0.000
8,C_perrieri_A12,0.000
9,C_pervilleana_A957,0.000


In [8]:
# binary classification: 0 if caffeine_percent is 0 (or below), 1 if above 0
caffeine_content['caffeine_class'] = (caffeine_content['caffeine_percent'] > 0).astype(int)

print(caffeine_content)


               Species_name  caffeine_percent  caffeine_class
0   C_andrambovatensis_A310             0.000               0
1          C_abbayesii_A601             0.000               0
2         C_arenesiana_A403             0.000               0
3           C_bertrandii_A5             0.000               0
4           C_dubardii_A969             0.000               0
..                      ...               ...             ...
20          C_vianneyi_A946             0.040               1
21   C_farafanganensis_A208             0.045               1
22          C_homollei_A945             0.060               1
23    C_kianjavatensis_A602             0.700               1
24        C_lancifolia_A320             0.700               1

[25 rows x 3 columns]


In [11]:
coffee_data['extracted_specimen'] = coffee_data['specimen_name'].str.split('_').str[1]
print(coffee_data.head)

<bound method NDFrame.head of      source_crs  longitude   latitude                              mada_geom_point  sampled_layers  nodata_layers  is_categorical_encoded         specimen_name  clim_1_tmin1_jan  clim_2_tmin2_feb  ...  env_74_solrad  env_75_geo  env_76_soi  env_77_veg  env_78_wat  env_79_forcov  clim_1_tmin1_jan_old  clim_13_tmax1_jan_old  clim_25_prec1_jan_old  extracted_specimen
0          4326  46.450000 -24.366660  POINT (647070.6204618097 7304410.220689219)              79              0                   False      Coffea_abbayesii             195.5             195.0  ...         7430.0        10.0         6.0         7.0         6.0            0.0                 159.0                  307.0                   30.0           abbayesii
1          4326  46.830000 -24.730000  POINT (685086.8720605411 7263711.610572983)              79              0                   False      Coffea_abbayesii             177.5             177.0  ...         7630.0        10.0         

In [12]:
caffeine_content['extracted_species'] = caffeine_content['Species_name'].str.split('_').str[1]
print(caffeine_content)

               Species_name  caffeine_percent  caffeine_class extracted_species
0   C_andrambovatensis_A310             0.000               0  andrambovatensis
1          C_abbayesii_A601             0.000               0         abbayesii
2         C_arenesiana_A403             0.000               0        arenesiana
3           C_bertrandii_A5             0.000               0        bertrandii
4           C_dubardii_A969             0.000               0          dubardii
..                      ...               ...             ...               ...
20          C_vianneyi_A946             0.040               1          vianneyi
21   C_farafanganensis_A208             0.045               1   farafanganensis
22          C_homollei_A945             0.060               1          homollei
23    C_kianjavatensis_A602             0.700               1    kianjavatensis
24        C_lancifolia_A320             0.700               1        lancifolia

[25 rows x 4 columns]


In [17]:
merged_df = pd.merge(
    coffee_data,
    caffeine_content[['extracted_species', 'Species_name', 'caffeine_percent','caffeine_class']],
    left_on='extracted_specimen',
    right_on='extracted_species',
    how='left'  # Use 'left' join to keep all rows from coffee_env_data
)

# Step 4: Drop the helper columns
merged_df = merged_df.drop(columns=['extracted_specimen', 'extracted_species'])

# The result is the original DataFrame with an additional 'caffeine_class' column
merged_df.to_csv(r'../data/coll_caff_node_w_class.csv', index=False)

In [18]:
merged_df

,source_crs,longitude,latitude,mada_geom_point,sampled_layers,nodata_layers,is_categorical_encoded,specimen_name,clim_1_tmin1_jan,clim_2_tmin2_feb,...,env_76_soi,env_77_veg,env_78_wat,env_79_forcov,clim_1_tmin1_jan_old,clim_13_tmax1_jan_old,clim_25_prec1_jan_old,Species_name,caffeine_percent,caffeine_class
0,4326,46.450000,-24.366660,POINT (647070.6204618097 7304410.220689219),79,0,False,Coffea_abbayesii,195.5,195.0,...,6.0,7.0,6.0,0.0,159.0,307.0,30.0,C_abbayesii_A601,0.00,0.0
1,4326,46.830000,-24.730000,POINT (685086.8720605411 7263711.610572983),79,0,False,Coffea_abbayesii,177.5,177.0,...,9.0,14.0,5.0,100.0,139.0,260.0,52.0,C_abbayesii_A601,0.00,0.0
2,4326,46.833300,-24.733300,POINT (685415.815086571 7263341.630764661),79,0,False,Coffea_abbayesii,177.5,177.0,...,9.0,14.0,5.0,100.0,139.0,260.0,52.0,C_abbayesii_A601,0.00,0.0
3,4326,46.833330,-24.666670,POINT (685517.5605412831 7270721.614432366),79,0,False,Coffea_abbayesii,208.5,208.0,...,10.0,7.0,5.0,0.0,168.0,292.0,61.0,C_abbayesii_A601,0.00,0.0
4,4326,46.833333,-24.733333,POINT (685419.1044266553 7263337.930920706),79,0,False,Coffea_abbayesii,177.5,177.0,...,9.0,14.0,5.0,100.0,139.0,260.0,52.0,C_abbayesii_A601,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
463,4326,47.866667,-21.383333,POINT (797253.6520891015 7632714.03380194),79,0,False,Coffea_vatovavyensis,216.0,215.0,...,11.0,16.0,3.0,0.0,178.0,276.0,71.0,C_vatovavyensis_A830,0.01,1.0
464,4326,47.933333,-21.400000,POINT (804136.3625319668 7630739.814647073),79,0,False,Coffea_vatovavyensis,208.0,207.0,...,11.0,16.0,3.0,18.0,170.0,267.0,69.0,C_vatovavyensis_A830,0.01,1.0
465,4326,47.863972,-21.382389,POINT (796975.9344399704 7632823.713043112),79,0,False,Coffea_vianneyi,211.0,210.0,...,11.0,16.0,3.0,0.0,173.0,273.0,69.0,C_vianneyi_A946,0.04,1.0
466,4326,47.866667,-21.383333,POINT (797253.6520891015 7632714.03380194),79,0,False,Coffea_vianneyi,216.0,215.0,...,11.0,16.0,3.0,0.0,178.0,276.0,71.0,C_vianneyi_A946,0.04,1.0


In [23]:
input_df = merged_df[["Species_name","longitude","latitude","caffeine_percent"]]
input_df = input_df.rename(columns={'Species_name': 'specimen_id'})
input_df.to_csv("../../data/coords_w_caff.csv", index=False)
input_df

,specimen_id,longitude,latitude,caffeine_percent
0,C_abbayesii_A601,46.450000,-24.366660,0.00
1,C_abbayesii_A601,46.830000,-24.730000,0.00
2,C_abbayesii_A601,46.833300,-24.733300,0.00
3,C_abbayesii_A601,46.833330,-24.666670,0.00
4,C_abbayesii_A601,46.833333,-24.733333,0.00
...,...,...,...,...
463,C_vatovavyensis_A830,47.866667,-21.383333,0.01
464,C_vatovavyensis_A830,47.933333,-21.400000,0.01
465,C_vianneyi_A946,47.863972,-21.382389,0.04
466,C_vianneyi_A946,47.866667,-21.383333,0.04
